In [0]:
%run ./00_retail_config


In [0]:
# pre-flight validation
# Check if the volume is accessible
for table, settings in INGESTION_CONFIG.items():
    try:
        if settings["type"] == "table":
            # Delta table — verify it exists and is readable
            count = spark.read.table(settings["source"]).limit(1).count()
            print(f"Table found for {table}: {settings['source']} — accessible")
        else:
            # Volume — verify the path exists
            files = dbutils.fs.ls(settings["source"])
            print(f"Path found for {table}: {len(files)} files detected.")
    except Exception as e:
        print(f"Access error for {table}: {e}")


In [0]:
# Check the schema and row count
for table, settings in INGESTION_CONFIG.items():
    print(f"--- Validating Schema: {table} ({settings['type']}) ---")
    if settings["type"] == "table":
        df = spark.read.table(settings["source"])
    else:
        df = (spark.read
              .format(settings["format"])
              .option("header", "true" if settings["format"] == "csv" else "false")
              .option("inferSchema", "true")
              .load(settings["source"]))
    df.printSchema()
    print(f"Total rows: {df.count()}")
    print(f"Total columns: {len(df.columns)}")


In [0]:
# select 5 records per schema
for table, settings in INGESTION_CONFIG.items():
    print(f"--- Selecting into {table} ({settings['type']}) ---")
    if settings["type"] == "table":
        df = spark.read.table(settings["source"]).limit(5)
    else:
        df = (spark.read
              .format(settings["format"])
              .option("header", "true" if settings["format"] == "csv" else "false")
              .option("inferSchema", "true")
              .load(settings["source"])
              .limit(5))
    display(df)

In [0]:
# Start of Data Ingestion
from pyspark.sql.functions import current_timestamp, lit

for table_name, settings in INGESTION_CONFIG.items():
    print(f"--- Starting Ingestion: {table_name} ---")

    if settings["type"] == "table":
        # Delta table source — Autoloader is for files only, use batch read instead
        (spark.read.table(settings["source"])
             .withColumn("_source_table", lit(settings["source"]))
             .withColumn("_ingested_at", current_timestamp())
             .write
             .format("delta")
             .mode("overwrite")
             .option("overwriteSchema", "true")
             .saveAsTable(settings["target"]))
    else:
        # Volume / file source — use Autoloader (cloudFiles)
        checkpoint = f"{CHECKPOINT_PATH}/{table_name}"
        schema_path = f"{CHECKPOINT_PATH}/{table_name}/schema"

        reader = (spark.readStream
                  .format("cloudFiles")
                  .option("cloudFiles.format", settings["format"])
                  .option("cloudFiles.inferColumnTypes", "true")
                  .option("cloudFiles.schemaLocation", schema_path))

        if settings["format"] == "csv":
            reader = reader.option("header", "true")

        (reader.load(settings["source"])
               .select("*", "_metadata.file_path", current_timestamp().alias("_ingested_at"))
               .writeStream
               .option("checkpointLocation", checkpoint)
               .option("mergeSchema", "true")
               .trigger(availableNow=True)
               .toTable(settings["target"])
               .awaitTermination())

    print(f"Success: {table_name} ingested into {settings['target']}")

In [0]:
# Validation Report
from pyspark.sql.functions import count

print("--- INGESTION REPORT ---")
for table, settings in INGESTION_CONFIG.items():
    try:
        record_count = spark.read.table(settings["target"]).count()
        print(f"Table: {settings['target']} | Total Rows: {record_count}")
    except Exception as e:
        print(f"Table: {settings['target']} | FAILED TO READ — {e}")
     